# SPARC Example 16: Enclosed Mass Profile M(<R)

**EPS Research RAG Astrophysics Corpus — Unified HI Corpus v7.0**

From the rotation curve we can compute the enclosed mass:

    M(<R) = V_obs^2 * R / G

where G = 4.301e-3 pc Msun^-1 (km/s)^2.
This gives total mass (baryons + dark matter) as a function of radius.

**Important note on corpus fidelity:** The `rotation_curve_corpus_v7_flat.csv` and `rotation_curve_corpus_v7.json` are **full-fidelity** — not a summary or veneer. The CSV contains every kinematic parameter published by Lelli et al. (2016) including per-galaxy inclination, distance uncertainties, mass-to-light ratios, and rotation curve statistics. The JSON adds full per-ring data: Vobs, Vgas, Vdisk, Vbul, errV at every radial point. This is the complete published dataset in a single machine-readable file.

**Corpus:** Flynn (2026), Zenodo DOI: 10.5281/zenodo.21768240  
**Source:** Lelli, McGaugh & Schombert (2016), AJ 152, 157  
**Dependencies:** Python 3, numpy, matplotlib, csv (standard library only)

In [ ]:
# ── Colab setup: auto-download corpus from Zenodo ─────────────
import os, sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    import urllib.request
    CORPORA = {
        'rotation_curve_corpus_v7.json': 'https://zenodo.org/records/21768240/files/rotation_curve_corpus_v7.json',
    }
    for filename, url in CORPORA.items():
        if not os.path.exists(filename):
            print(f"Downloading {filename}...")
            urllib.request.urlretrieve(url, filename)
            print(f"  ✓ {filename}")
        else:
            print(f"  Already present: {filename}")
    print("Ready.")
else:
    print("Running locally — corpus files loaded from working directory.")


In [ ]:
import json
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

with open('rotation_curve_corpus_v7.json') as f:
    corpus = json.load(f)

# G in kpc (km/s)^2 Msun^-1
G_kpc = 4.301e-3 * 1e-3

targets = ['DDO161', 'NGC2403', 'UGC2885']
colors  = ['#2166ac', '#4dac26', '#d6604d']

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Panel 1: Enclosed mass profiles
ax = axes[0]
for name, color in zip(targets, colors):
    try:
        g = next(g for g in corpus['galaxies'] if g['galaxy'] == name)
        d = g['data']
        R    = np.array([p['Rad']  for p in d])
        Vobs = np.array([p['Vobs'] for p in d])
        M_enc = Vobs**2 * R / G_kpc  # solar masses
        ax.plot(R, np.log10(M_enc), 'o-', color=color,
                ms=5, lw=1.5, label=name)
    except StopIteration:
        pass

ax.set_xlabel('Radius (kpc)', fontsize=12)
ax.set_ylabel(r'log $M_{enc}$ ($M_\odot$)', fontsize=12)
ax.set_title('Enclosed Mass Profiles\nM(R) = V²R/G', fontsize=11)
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

# Panel 2: Normalized profiles
ax2 = axes[1]
for name, color in zip(targets, colors):
    try:
        g = next(g for g in corpus['galaxies'] if g['galaxy'] == name)
        d = g['data']
        R    = np.array([p['Rad']  for p in d])
        Vobs = np.array([p['Vobs'] for p in d])
        M_enc = Vobs**2 * R / G_kpc
        ax2.plot(R/R[-1], M_enc/M_enc[-1], 'o-', color=color,
                 ms=5, lw=1.5, label=name)
    except StopIteration:
        pass

ax2.set_xlabel(r'R / R$_{max}$', fontsize=12)
ax2.set_ylabel(r'M$_{enc}$ / M$_{enc,max}$', fontsize=12)
ax2.set_title('Normalized Enclosed Mass\n(concentration comparison)', fontsize=11)
ax2.legend(fontsize=9)
ax2.grid(alpha=0.3)

plt.suptitle('HI Corpus v7.0 — Enclosed Mass Profiles', fontsize=11)
plt.tight_layout()
plt.savefig('ex16_enclosed_mass.png', dpi=150, bbox_inches='tight')
plt.show()
